# Whisper Diarizer

In [4]:
# @title Download dependences

import importlib.util
import subprocess

def install_if_missing(package, install_name=None):
    """
    Checks if a package is installed, and installs it if missing.

    Parameters:
    - package: The module name used in Python (e.g., "torch")
    - install_name: The name used for installation (e.g., "torch" or "git+https://github.com/openai/whisper.git")
    """
    if importlib.util.find_spec(package) is None:
        install_name = install_name or package
        print(f"Installing {install_name}...")
        subprocess.call(["pip", "install", install_name])
    else:
        print(f"{package} is already installed.")

# Check and install packages
install_if_missing("whisper", "git+https://github.com/openai/whisper.git")
install_if_missing("speechbrain")
install_if_missing("torch")
install_if_missing("torchvision")
install_if_missing("torchaudio")
install_if_missing("numpy")
install_if_missing("ffmpeg")
install_if_missing("pyannote", "git+https://github.com/pyannote/pyannote-audio")  # Changed from "pyannote.audio" to "pyannote"

whisper is already installed.
speechbrain is already installed.
torch is already installed.
torchvision is already installed.
torchaudio is already installed.
numpy is already installed.
ffmpeg is already installed.
pyannote is already installed.


In [7]:
# @title Main Code

import os
import subprocess
import datetime
import torch
import numpy as np
import whisper
import wave
import contextlib
from pyannote.audio import Audio
from pyannote.audio.pipelines.speaker_verification import PretrainedSpeakerEmbedding
from pyannote.core import Segment
from sklearn.cluster import AgglomerativeClustering

# Upload the file
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))

print(f"Uploaded file: {path}")

# Check if the file is actually a WAV file
def is_wav_file(filepath):
    try:
        with wave.open(filepath, 'r') as f:
            print(f"✅ {filepath} is a valid WAV file.")
            return True
    except wave.Error as e:
        print(f"❌ {filepath} is NOT a valid WAV file: {e}")
        return False

# Convert non-WAV files to WAV
converted_path = "converted_audio.wav"
if not is_wav_file(path):
    print("🔄 Converting file to WAV format...")
    subprocess.call(['ffmpeg', '-i', path, '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', converted_path, '-y'])
    path = converted_path

# Verify conversion
if not is_wav_file(path):
    raise ValueError("🚨 Conversion failed! The file is not a valid WAV format.")

print(f"✅ Using file: {path}")

# User-configurable parameters
num_speakers = 3  # Change to None for automatic estimation
language = 'any'  # Can be 'any' or 'English'
model_size = 'large-v2'  # ['tiny', 'base', 'small', 'medium', 'large']

# Load Whisper model
print("⏳ Loading Whisper model...")
model = whisper.load_model(model_size)
print("✅ Whisper model loaded.")

# Transcribe with automatic language detection if 'any' is selected
print("⏳ Transcribing audio...")
result = model.transcribe(path, language=None if language == "any" else "en")
segments = result.get("segments", [])

if not segments:
    raise ValueError("🚨 No speech detected in the audio file!")

print(f"✅ Transcription complete. Detected {len(segments)} segments.")

# Determine audio duration
with contextlib.closing(wave.open(path, 'r')) as f:
    frames = f.getnframes()
    rate = f.getframerate()
    duration = frames / float(rate)

print(f"📏 Audio duration: {duration:.2f} seconds")

# Load speaker embedding model
print("⏳ Loading speaker embedding model...")
embedding_model = PretrainedSpeakerEmbedding("speechbrain/spkrec-ecapa-voxceleb", device=torch.device("cuda"))
audio = Audio()
print("✅ Speaker embedding model loaded.")

# Function to extract speaker embeddings
def segment_embedding(segment):
    try:
        start, end = segment["start"], min(duration, segment["end"])
        clip = Segment(start, end)
        waveform, sample_rate = audio.crop(path, clip)

        if not isinstance(waveform, torch.Tensor):
            waveform = torch.tensor(waveform)

        if waveform.ndim == 1:  # Convert 1D tensor to 3D
            waveform = waveform.unsqueeze(0).unsqueeze(0)
        elif waveform.ndim == 2:  # Convert [channels, samples] to [1, channels, samples]
            waveform = waveform.unsqueeze(0)

        if waveform.shape[1] > 1:
            waveform = waveform.mean(dim=1, keepdim=True)

        waveform = waveform.to(torch.float32)

        assert waveform.shape[1] == 1, "🚨 Waveform must be mono (1 channel)"

        return embedding_model(waveform)
    except Exception as e:
        print(f"🚨 Error processing segment {segment['start']} - {segment['end']}: {e}")
        return np.zeros(192)  # Return a dummy embedding

# Compute embeddings
embeddings = np.zeros((len(segments), 192))
for i, segment in enumerate(segments):
    print(f"🎤 Processing segment {i+1}/{len(segments)}: {segment['start']} - {segment['end']}")
    embeddings[i] = segment_embedding(segment)

# Handle NaN values
embeddings = np.nan_to_num(embeddings)

# Perform speaker clustering
print("⏳ Performing speaker clustering...")
clustering = AgglomerativeClustering(n_clusters=num_speakers).fit(embeddings)
labels = clustering.labels_
print("✅ Speaker clustering complete.")

# Assign speaker labels
for i in range(len(segments)):
    segments[i]["speaker"] = f"SPEAKER {labels[i] + 1}"

# Format timestamps
def format_time(seconds):
    return str(datetime.timedelta(seconds=round(seconds)))

# Save transcript
print("💾 Saving transcript...")
with open("transcript.txt", "w") as f:
    for i, segment in enumerate(segments):
        if i == 0 or segments[i - 1]["speaker"] != segment["speaker"]:
            f.write(f"\n{segment['speaker']} [{format_time(segment['start'])}]\n")
        f.write(segment["text"].strip() + " ")

print("✅ Transcription completed! Saved as transcript.txt.")

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

# Empty /content with exceptions 🕳️

In [ ]:
import os
import shutil

# Define the allowed file extensions
allowed_extensions = {'.mp3', '.[file_type]'}

# Get a list of all files and folders in /content
content_path = "/content"
for item in os.listdir(content_path):
    item_path = os.path.join(content_path, item)

    # If it's a file and not in allowed extensions, delete it
    if os.path.isfile(item_path):
        if not any(item.endswith(ext) for ext in allowed_extensions):
            os.remove(item_path)
            print(f"Deleted file: {item_path}")

    # If it's a directory, delete the entire folder and its contents
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)
        print(f"Deleted folder: {item_path}")

print("Cleanup complete!")

Deleted folder: /content/.config
Deleted file: /content/IELTS Listening Practice Test with Answers 🎧 English Listening Practice.mp4
Deleted file: /content/IELTS Listening Practice Test with Answers 🎧 English Listening Practice (1).mp4
Deleted file: /content/converted_audio.wav
Deleted file: /content/transcript.txt
Deleted folder: /content/.ipynb_checkpoints
Deleted folder: /content/sample_data
Cleanup complete!


In [ ]:

# @title Enhanced Universal Audio/Video Transcription & Diarization
# This script processes ALL audio/video files from /content folder automatically

import importlib.util
import subprocess
import os
import glob
import datetime
import torch
import numpy as np
import whisper
import wave
import contextlib
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

def install_if_missing(package, install_name=None):
    """Install package if not already installed"""
    if importlib.util.find_spec(package) is None:
        install_name = install_name or package
        print(f"📦 Installing {install_name}...")
        try:
            subprocess.check_call(["pip", "install", install_name, "-q"])
            print(f"✅ {package} installed successfully")
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to install {package}: {e}")
            return False
    else:
        print(f"✅ {package} already installed")
    return True

# Install dependencies
print("🔧 Checking and installing dependencies...")
dependencies = [
    ("whisper", "git+https://github.com/openai/whisper.git"),
    ("speechbrain", None),
    ("torch", None),
    ("torchvision", None),
    ("torchaudio", None),
    ("numpy", None),
    ("sklearn", "scikit-learn"),
    ("pyannote.audio", "git+https://github.com/pyannote/pyannote-audio"),
    ("ffmpeg_python", "ffmpeg-python")
]

for package, install_name in dependencies:
    install_if_missing(package, install_name)

# Import required libraries
try:
    from pyannote.audio import Audio
    from pyannote.audio.pipelines.speaker_verification import PretrainedSpeakerEmbedding
    from pyannote.core import Segment
    from sklearn.cluster import AgglomerativeClustering
    import ffmpeg
    print("✅ All libraries imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("🔄 Trying alternative import method...")

class AudioVideoProcessor:
    def __init__(self):
        self.supported_formats = {
            'audio': ['.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.wma', '.opus'],
            'video': ['.mp4', '.avi', '.mkv', '.mov', '.wmv', '.flv', '.webm', '.m4v', '.3gp']
        }
        self.whisper_model = None
        self.embedding_model = None
        self.audio_processor = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"🖥️  Using device: {self.device}")

    def load_models(self, model_size='large-v2'):
        """Load Whisper and speaker embedding models"""
        try:
            print(f"⏳ Loading Whisper model ({model_size})...")
            self.whisper_model = whisper.load_model(model_size)
            print("✅ Whisper model loaded")

            print("⏳ Loading speaker embedding model...")
            self.embedding_model = PretrainedSpeakerEmbedding(
                "speechbrain/spkrec-ecapa-voxceleb",
                device=self.device
            )
            self.audio_processor = Audio()
            print("✅ Speaker embedding model loaded")

        except Exception as e:
            print(f"❌ Error loading models: {e}")
            raise

    def find_media_files(self, directory="/content"):
        """Find media files, prioritizing existing WAV files in /content/processed"""
        processed_dir = "/content/processed"

        # First, check for existing WAV files in processed directory
        wav_files = []
        if os.path.exists(processed_dir):
            print(f"🔍 Checking {processed_dir} for existing WAV files...")
            wav_pattern = os.path.join(processed_dir, "*.wav")
            wav_files = glob.glob(wav_pattern)

            if wav_files:
                wav_files = sorted(wav_files)
                print(f"✅ Found {len(wav_files)} existing WAV files in processed directory:")
                for file in wav_files:
                    print(f"   🎵 {os.path.basename(file)}")
                return wav_files, True  # Return True to indicate these are already processed

        # If no WAV files found, scan for other media files to convert
        print(f"🔍 No WAV files found in {processed_dir}, scanning {directory} for media files to convert...")

        media_files = []
        all_formats = self.supported_formats['audio'] + self.supported_formats['video']

        for ext in all_formats:
            pattern = os.path.join(directory, f"**/*{ext}")
            files = glob.glob(pattern, recursive=True)
            media_files.extend(files)

        # Remove duplicates and sort
        media_files = sorted(list(set(media_files)))

        print(f"📁 Found {len(media_files)} media files to convert:")
        for file in media_files:
            print(f"   📄 {os.path.basename(file)}")

        return media_files, False  # Return False to indicate these need conversion

    def convert_to_wav(self, input_file, output_dir="/content/processed"):
        """Convert any audio/video file to WAV format"""
        os.makedirs(output_dir, exist_ok=True)

        filename = Path(input_file).stem
        output_file = os.path.join(output_dir, f"{filename}.wav")

        try:
            print(f"🔄 Converting {os.path.basename(input_file)} to WAV...")

            # Use ffmpeg to convert with optimal settings
            (
                ffmpeg
                .input(input_file)
                .output(
                    output_file,
                    acodec='pcm_s16le',
                    ac=1,  # mono
                    ar='16000'  # 16kHz sample rate
                )
                .overwrite_output()
                .run(quiet=True, capture_stdout=True)
            )

            print(f"✅ Converted to: {output_file}")
            return output_file

        except ffmpeg.Error as e:
            print(f"❌ FFmpeg error converting {input_file}: {e}")
            return None
        except Exception as e:
            print(f"❌ Error converting {input_file}: {e}")
            return None

    def detect_language(self, audio_file, sample_duration=30):
        """Detect language using Whisper"""
        try:
            print("🌍 Detecting language...")

            # Load a small sample for language detection
            audio = whisper.load_audio(audio_file)
            audio = whisper.pad_or_trim(audio, length=sample_duration * 16000)

            # Make log-Mel spectrogram and move to the same device as the model
            mel = whisper.log_mel_spectrogram(audio).to(self.whisper_model.device)

            # Detect the spoken language
            _, probs = whisper.detect_language(self.whisper_model, mel)
            detected_language = max(probs, key=probs.get)
            confidence = probs[detected_language]

            print(f"🗣️  Detected language: {detected_language} (confidence: {confidence:.2f})")
            return detected_language, confidence

        except Exception as e:
            print(f"❌ Language detection failed: {e}")
            return "en", 0.5  # Default to English

    def transcribe_audio(self, audio_file, language=None):
        """Transcribe audio using Whisper"""
        try:
            print("⏳ Transcribing audio...")

            # Transcribe with detected language or auto-detect
            result = self.whisper_model.transcribe(
                audio_file,
                language=language,
                task="transcribe",
                verbose=False
            )

            segments = result.get("segments", [])
            if not segments:
                print("❌ No speech detected in audio")
                return None, None

            print(f"✅ Transcription complete: {len(segments)} segments")
            return result, segments

        except Exception as e:
            print(f"❌ Transcription failed: {e}")
            return None, None

    def get_audio_duration(self, audio_file):
        """Get audio file duration"""
        try:
            with contextlib.closing(wave.open(audio_file, 'r')) as f:
                frames = f.getnframes()
                rate = f.getframerate()
                duration = frames / float(rate)
            return duration
        except Exception as e:
            print(f"❌ Error getting audio duration: {e}")
            return 0

    def extract_speaker_embedding(self, audio_file, segment, duration):
        """Extract speaker embedding for a segment"""
        try:
            start, end = segment["start"], min(duration, segment["end"])
            clip = Segment(start, end)

            waveform, sample_rate = self.audio_processor.crop(audio_file, clip)

            # Ensure proper tensor format
            if not isinstance(waveform, torch.Tensor):
                waveform = torch.tensor(waveform)

            if waveform.ndim == 1:
                waveform = waveform.unsqueeze(0).unsqueeze(0)
            elif waveform.ndim == 2:
                waveform = waveform.unsqueeze(0)

            if waveform.shape[1] > 1:
                waveform = waveform.mean(dim=1, keepdim=True)

            waveform = waveform.to(torch.float32)

            # Get embedding and handle both tensor and numpy returns
            embedding = self.embedding_model(waveform)

            # Convert to numpy if it's a tensor, otherwise use as-is
            if isinstance(embedding, torch.Tensor):
                return embedding.cpu().numpy()
            else:
                # Already a numpy array
                return np.array(embedding)

        except Exception as e:
            # Silent error handling - will be reported in summary
            return np.zeros(192)  # Return dummy embedding

    def perform_speaker_diarization(self, audio_file, segments, num_speakers=None):
        """Perform speaker diarization"""
        try:
            print("⏳ Performing speaker diarization...")

            duration = self.get_audio_duration(audio_file)
            embeddings = np.zeros((len(segments), 192))

            # Extract embeddings for each segment
            error_count = 0
            for i, segment in enumerate(segments):
                if (i + 1) % 50 == 0:  # Reduced frequency of progress messages
                    print(f"🎤 Processing segment {i+1}/{len(segments)}")

                embedding = self.extract_speaker_embedding(audio_file, segment, duration)
                embeddings[i] = embedding

                # Count failed embeddings (all zeros)
                if np.allclose(embedding, 0):
                    error_count += 1

            # Report summary instead of individual errors
            if error_count > 0:
                success_rate = (len(segments) - error_count) / len(segments) * 100
                print(f"⚠️  {error_count}/{len(segments)} segments had embedding errors (Success rate: {success_rate:.1f}%)")

            # Handle NaN values
            embeddings = np.nan_to_num(embeddings)

            # Determine optimal number of speakers if not provided
            if num_speakers is None:
                # Simple heuristic: use elbow method or default based on duration
                max_speakers = min(8, max(2, len(segments) // 20))
                num_speakers = min(max_speakers, 4)  # Cap at 4 for reasonable processing
                print(f"🤖 Auto-detected {num_speakers} speakers")

            # Perform clustering only if we have valid embeddings
            if not np.allclose(embeddings, 0):
                clustering = AgglomerativeClustering(n_clusters=num_speakers)
                labels = clustering.fit_predict(embeddings)
            else:
                print("⚠️  All embeddings failed, assigning single speaker")
                labels = np.zeros(len(segments), dtype=int)

            # Assign speaker labels
            for i, label in enumerate(labels):
                segments[i]["speaker"] = f"SPEAKER_{label + 1:02d}"

            print("✅ Speaker diarization complete")
            return segments

        except Exception as e:
            print(f"❌ Speaker diarization failed: {e}")
            # Assign single speaker to all segments
            for segment in segments:
                segment["speaker"] = "SPEAKER_01"
            return segments

    def save_transcript(self, segments, output_file, metadata=None):
        """Save transcript in multiple formats"""
        try:
            base_name = Path(output_file).stem
            output_dir = Path(output_file).parent

            # Save as TXT
            txt_file = output_dir / f"{base_name}_transcript.txt"
            with open(txt_file, "w", encoding="utf-8") as f:
                if metadata:
                    f.write(f"Transcription Metadata:\n")
                    f.write(f"File: {metadata.get('filename', 'Unknown')}\n")
                    f.write(f"Language: {metadata.get('language', 'Unknown')}\n")
                    f.write(f"Duration: {metadata.get('duration', 'Unknown')}\n")
                    f.write(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                    f.write("="*50 + "\n\n")

                current_speaker = None
                for segment in segments:
                    if current_speaker != segment["speaker"]:
                        current_speaker = segment["speaker"]
                        timestamp = self.format_time(segment["start"])
                        f.write(f"\n{current_speaker} [{timestamp}]\n")
                    f.write(segment["text"].strip() + " ")

            # Save as JSON
            json_file = output_dir / f"{base_name}_transcript.json"
            transcript_data = {
                "metadata": metadata or {},
                "segments": segments
            }
            with open(json_file, "w", encoding="utf-8") as f:
                json.dump(transcript_data, f, indent=2, ensure_ascii=False)

            print(f"✅ Transcript saved:")
            print(f"   📄 TXT: {txt_file}")
            print(f"   📄 JSON: {json_file}")

            return str(txt_file), str(json_file)

        except Exception as e:
            print(f"❌ Error saving transcript: {e}")
            return None, None

    def format_time(self, seconds):
        """Format time in HH:MM:SS"""
        return str(datetime.timedelta(seconds=round(seconds)))

    def process_file(self, file_path, num_speakers=None, already_wav=False):
        """Process a single media file"""
        print(f"\n{'='*60}")
        print(f"🎯 Processing: {os.path.basename(file_path)}")
        print(f"{'='*60}")

        try:
            # Use existing WAV or convert to WAV
            if already_wav:
                print(f"✅ Using existing WAV file: {file_path}")
                wav_file = file_path
            else:
                wav_file = self.convert_to_wav(file_path)
                if not wav_file:
                    return False

            # Detect language
            language, confidence = self.detect_language(wav_file)

            # Transcribe
            result, segments = self.transcribe_audio(wav_file, language)
            if not segments:
                return False

            # Perform speaker diarization
            segments = self.perform_speaker_diarization(wav_file, segments, num_speakers)

            # Prepare metadata
            duration = self.get_audio_duration(wav_file)
            metadata = {
                "original_filename": os.path.basename(file_path),
                "wav_file": os.path.basename(wav_file),
                "already_converted": already_wav,
                "language": language,
                "language_confidence": confidence,
                "duration": self.format_time(duration),
                "num_segments": len(segments),
                "model_used": "large-v2"
            }

            # Save transcript
            if already_wav:
                # Use the WAV filename for output
                output_base = os.path.join("/content/transcripts", Path(wav_file).stem)
            else:
                # Use original filename for output
                output_base = os.path.join("/content/transcripts", Path(file_path).stem)

            os.makedirs("/content/transcripts", exist_ok=True)

            txt_file, json_file = self.save_transcript(segments, output_base, metadata)

            print(f"✅ Successfully processed {os.path.basename(file_path)}")
            return True

        except Exception as e:
            print(f"❌ Failed to process {file_path}: {e}")
            return False

    def process_all_files(self, directory="/content", num_speakers=None):
        """Process all media files in directory, prioritizing existing WAV files"""
        print("🚀 Starting Universal Media Transcription & Diarization")
        print("="*60)

        # Find media files (prioritizing existing WAV files)
        media_files, already_wav = self.find_media_files(directory)
        if not media_files:
            print("❌ No media files found in the directory")
            return

        # Load models
        self.load_models()

        # Process each file
        successful = 0
        failed = 0

        if already_wav:
            print("🎵 Processing existing WAV files from /content/processed/")
        else:
            print("🔄 Processing and converting media files from /content/")

        for file_path in media_files:
            if self.process_file(file_path, num_speakers, already_wav):
                successful += 1
            else:
                failed += 1

        # Summary
        print(f"\n{'='*60}")
        print(f"📊 PROCESSING SUMMARY")
        print(f"{'='*60}")
        print(f"✅ Successfully processed: {successful} files")
        print(f"❌ Failed: {failed} files")
        print(f"📁 Transcripts saved in: /content/transcripts/")

        if successful > 0:
            print(f"\n🎉 All transcripts are ready!")
            print(f"📄 Check /content/transcripts/ for your files")
            if not already_wav:
                print(f"🎵 Converted WAV files saved in: /content/processed/")
                print(f"💡 Next time, these WAV files will be used directly for faster processing!")

# Initialize and run the processor
if __name__ == "__main__":
    processor = AudioVideoProcessor()

    # Configuration
    NUM_SPEAKERS = None  # Set to None for auto-detection, or specify number
    SOURCE_DIRECTORY = "/content"  # Directory to scan for media files

    # Process all files
    processor.process_all_files(SOURCE_DIRECTORY, NUM_SPEAKERS)

🔧 Checking and installing dependencies...
✅ whisper already installed
✅ speechbrain already installed
✅ torch already installed
✅ torchvision already installed
✅ torchaudio already installed
✅ numpy already installed
✅ sklearn already installed
✅ pyannote.audio already installed
📦 Installing ffmpeg-python...
✅ ffmpeg_python installed successfully
✅ All libraries imported successfully
🖥️  Using device: cuda
🚀 Starting Universal Media Transcription & Diarization
🔍 No WAV files found in /content/processed, scanning /content for media files to convert...
📁 Found 0 media files to convert:
❌ No media files found in the directory
